# Ask Coresight: train the search model (short run, about 12 minutes)

The first run measured four versions; **3 epochs** was best (unseen 50.8%, golden 66.9%, against 39.0% / 58.3% today).
This run trains only that one, shrinks it for the server, checks it, and downloads **bge-cip.zip**.

1. Runtime → Change runtime type → **A100** or **L4** GPU → Save.
2. Runtime → **Run all**; at **Choose Files** pick `catalogue.json`.
3. **Keep the laptop open and awake until the zip downloads** (closing it stops the run).

In [ ]:
# quiet: Colab's own packages (gradio, diffusers) are not used here, so pip's notes about them are hidden
!pip -q install "sentence-transformers==3.3.1" onnx onnxscript onnxruntime "google-genai>=1.0" > /dev/null 2>&1   # 3.3.1: the version that trained here on 12 Nov
import torch, json, random, time, os, numpy as np
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none: Runtime → Change runtime type → GPU")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
from google.colab import files
if not os.path.exists("catalogue.json"):
    files.upload()   # choose catalogue.json
data = json.load(open("catalogue.json"))
docs, golden = data["docs"], data["golden"]
TRAIN_ASKS = 6   # asks 1-6 train and index; asks 7-8 are the unseen test
print(len(docs), "survey questions,", len(golden), "golden questions")

In [ ]:
from sentence_transformers import SentenceTransformer

def index_texts():
    owners, texts = [], []
    for n, d in enumerate(docs):
        for t in [d["embed"]] + d["asks"][:TRAIN_ASKS] + d.get("extra", []):
            owners.append(n); texts.append(t)
    return np.array(owners), texts

def score(encode, label):
    """Share right on unseen asks and golden, and at each confidence gap (best vs next *different* survey question)."""
    owners, texts = index_texts()
    vec = encode(texts)
    out = {}
    tests = {"unseen": ([a for d in docs for a in d["asks"][TRAIN_ASKS:]], [n for n, d in enumerate(docs) for _ in d["asks"][TRAIN_ASKS:]]),
             "golden": ([g["q"] for g in golden], golden)}
    for name, (queries, wants) in tests.items():
        sims = encode(queries) @ vec.T
        rows = []
        for row, want in zip(sims, wants):
            per = np.full(len(docs), -1.0); np.maximum.at(per, owners, row)
            order = np.argsort(-per)[:50]; top = order[0]
            other = next((o for o in order[1:] if docs[o]["text"].lower() != docs[top]["text"].lower()), order[1])
            right = docs[top]["text"].lower() == docs[want]["text"].lower() if name == "unseen" else any(w in docs[top]["text"].lower() for w in want["want"])
            rows.append((per[top] - per[other], right))
        rows = np.array(rows, float)
        curve = {g: (float(rows[rows[:, 0] >= g, 1].mean()) if (rows[:, 0] >= g).any() else 0.0, float((rows[:, 0] >= g).mean()))
                 for g in (0.0, 0.02, 0.04, 0.06, 0.08, 0.10, 0.15)}
        out[name] = curve
        print(f"{label:34} {name:7} right {curve[0.0][0]:.1%} · " + " · ".join(f"gap≥{g}: {r:.1%} on {c:.0%}" for g, (r, c) in list(curve.items())[1:]))
    return out

def st_encoder(model):
    return lambda texts: model.encode(texts, batch_size=256, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)

results = {}   # today's model was measured in the first run: unseen 39.0%, golden 58.3%

In [ ]:
# training pairs: each typed question (asks 1-6, name, aliases) with the survey question it means
anchors = [(t, n) for n, d in enumerate(docs) for t in d["asks"][:TRAIN_ASKS] + [d["name"]] + d["aliases"] if t]
hard = [n for _, n in anchors]   # no look-alikes in this run
print(len(anchors), "training pairs")

def train(epochs, with_hard, lr=3e-5, batch=128, scale=20.0):
    """In-batch contrastive training (MultipleNegativesRankingLoss written out): each typed question nearer its own survey
    question than every other in the batch, and than its hard look-alike when with_hard."""
    model = SentenceTransformer("BAAI/bge-small-en-v1.5", device=DEVICE)
    items = list(zip(anchors, hard))
    steps = []
    for e in range(epochs):
        random.Random(e).shuffle(items)
        current, groups = [], set()
        for (t, n), h in items:
            g = docs[n]["text"].lower()
            if g in groups: continue
            current.append((t, n, h)); groups.add(g)
            if len(current) == batch:
                steps.append(current); current, groups = [], set()
        if len(current) > 1:   # the last, part-full batch
            steps.append(current)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    warm = max(1, len(steps) // 10)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / warm) * max(0.0, 1 - s / len(steps)))
    model.train()
    emb = lambda ts: torch.nn.functional.normalize(model({k: v.to(DEVICE) for k, v in model.tokenize(ts).items() if torch.is_tensor(v)})["sentence_embedding"], dim=-1)
    started = time.time()
    for step in steps:
        a = emb([t for t, _, _ in step])
        targets = [docs[n]["embed"] for _, n, _ in step] + ([docs[h]["embed"] for _, _, h in step] if with_hard else [])
        p = emb(targets)
        loss = torch.nn.functional.cross_entropy(a @ p.T * scale, torch.arange(len(step), device=DEVICE))
        loss.backward(); opt.step(); sched.step(); opt.zero_grad()
    print(f"trained {epochs} epoch(s){' + look-alikes' if with_hard else ''} in {time.time() - started:.0f}s")
    model.eval()
    return model

models = {}
for epochs, with_hard in [(3, False)]:
    label = f"tuned {epochs} ep{' + look-alikes' if with_hard else ''}"
    models[label] = train(epochs, with_hard)
    results[label] = score(st_encoder(models[label]), label)

In [ ]:
# The best: most right on unseen wording and golden together (equal weight)
best = max(models, key=lambda k: results[k]["unseen"][0.0][0] + results[k]["golden"][0.0][0])
print("best:", best)
models[best].save("best")

In [ ]:
# Shrink to the server's format: ONNX, first-token vector, weights in 8 bits; checked with the server's own code path
import onnxruntime, tokenizers
from onnxruntime.quantization import quantize_dynamic, QuantType
bert = models[best][0].auto_model.cpu().eval()
dummy = models[best].tokenize(["a question"])
torch.onnx.export(bert, (dummy["input_ids"], dummy["attention_mask"], dummy["token_type_ids"]), "full.onnx",
                  input_names=["input_ids", "attention_mask", "token_type_ids"], output_names=["last_hidden_state"],
                  dynamic_axes={k: {0: "batch", 1: "tokens"} for k in ["input_ids", "attention_mask", "token_type_ids", "last_hidden_state"]},
                  opset_version=17, dynamo=False)   # the classic exporter: the same graph on every torch version
quantize_dynamic("full.onnx", "model_optimized.onnx", weight_type=QuantType.QInt8)
models[best].tokenizer.save_pretrained("tok")
os.replace("tok/tokenizer.json", "tokenizer.json")

words = tokenizers.Tokenizer.from_file("tokenizer.json")
words.enable_truncation(512); words.enable_padding(pad_id=0, pad_token="[PAD]")
options = onnxruntime.SessionOptions(); options.intra_op_num_threads = options.inter_op_num_threads = 1   # as on Azure
session = onnxruntime.InferenceSession("model_optimized.onnx", options, providers=["CPUExecutionProvider"])
def onnx_encode(texts):   # api/ask_model.py embed, line for line
    out = np.zeros((len(texts), 384), np.float32)
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))
    for s in range(0, len(order), 32):
        b = order[s:s + 32]; coded = words.encode_batch([texts[i] for i in b])
        ids = np.array([c.ids for c in coded], np.int64)
        out[b] = session.run(None, {"input_ids": ids, "attention_mask": np.array([c.attention_mask for c in coded], np.int64),
                                    "token_type_ids": np.zeros_like(ids)})[0][:, 0]
    return out / np.maximum(np.linalg.norm(out, axis=1, keepdims=True), 1e-9)
results[best + " · 8-bit ONNX"] = score(onnx_encode, best + " · 8-bit ONNX")
started = time.time(); [onnx_encode([g["q"]]) for g in golden]
print(f"one question on one CPU thread: {(time.time() - started) / len(golden) * 1000:.1f} ms · file {os.path.getsize('model_optimized.onnx') / 1e6:.0f} MB")

In [ ]:
import shutil
json.dump({"best": best, "results": {k: {t: {str(g): v for g, v in c.items()} for t, c in r.items()} for k, r in results.items()}},
          open("results.json", "w"), indent=1)
os.makedirs("bge-cip", exist_ok=True)
for f in ["model_optimized.onnx", "tokenizer.json", "results.json"] + (["catalogue_extra.json"] if os.path.exists("catalogue_extra.json") else []):
    shutil.copy(f, "bge-cip/")
shutil.make_archive("bge-cip", "zip", "bge-cip")
print("READY: bge-cip.zip is downloading to your Downloads folder.")
print("If the download does not start: left bar → folder icon → bge-cip.zip → ⋮ → Download (within 15 minutes).")
files.download("bge-cip.zip")

In [ ]:
# Stop paying once the download has had plenty of time: an idle GPU still uses units (15 min on an L4 ≈ 0.5)
import time
from google.colab import runtime
print("Releasing the GPU in 15 minutes. If bge-cip.zip is not in Downloads yet: folder icon → bge-cip.zip → ⋮ → Download")
time.sleep(900)
runtime.unassign()